In [18]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**A. Membaca Eksplorasi Awal**

In [30]:
import pyspark.pandas as ps

In [42]:
df = ps.read_csv("hdfs://localhost:9000//user/asfadani/tugas4/transaksi_september_2026.csv")
display(df.head(11))
print(df.info())
print("\nJumlah baris : \n",df.count())

/home/asfadani/anaconda3/envs/bigdata/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,order_id,tanggal,kategori,kota,unit_terjual,harga_satuan,metode_pembayaran,rating
0,ORD-3000,2026-09-02,Rumah Tangga,Yogyakarta,3,90000,COD,4.0
1,ORD-3001,2026-09-04,Makanan & Minuman,Solo,3,200000,E-Wallet,5.0
2,ORD-3002,2026-09-26,Kesehatan & Kecantikan,Semarang,8,60000,E-Wallet,3.0
3,ORD-3003,2026-09-09,Makanan & Minuman,Semarang,6,350000,Transfer Bank,4.0
4,ORD-3004,2026-09-10,Rumah Tangga,Yogyakarta,10,60000,E-Wallet,4.0
5,ORD-3005,2026-09-09,Fashion,Purworejo,5,20000,E-Wallet,4.0
6,ORD-3006,2026-09-19,Makanan & Minuman,Yogyakarta,2,20000,COD,5.0
7,ORD-3007,2026-09-05,Makanan & Minuman,Magelang,8,90000,Transfer Bank,NaN
8,ORD-3008,2026-09-06,Fashion,Semarang,7,20000,Kartu Kredit,5.0
9,ORD-3009,2026-09-21,Elektronik,Yogyakarta,10,90000,Transfer Bank,3.0


<class 'pyspark.pandas.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           1000 non-null   object        
 1   tanggal            1000 non-null   datetime64[ns]
 2   kategori           1000 non-null   object        
 3   kota               1000 non-null   object        
 4   unit_terjual       1000 non-null   int32         
 5   harga_satuan       1000 non-null   int32         
 6   metode_pembayaran  1000 non-null   object        
 7   rating             796 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int32(2), object(4)None

Jumlah baris : 
 order_id             1000
tanggal              1000
kategori             1000
kota                 1000
unit_terjual         1000
harga_satuan         1000
metode_pembayaran    1000
rating                796
dtype: int64


Dapat dilihat bahwa kolom rating memiliki nilai yang kosong (hanya 796 baris)

**B. Menangani Data Kosong**

In [44]:
print(df['rating'].describe())

print(df['rating'].mode())

count    796.000000
mean       4.145729
std        0.966805
min        1.000000
25%        4.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64
0    5.0
Name: rating, dtype: float64


In [49]:
avg_rating = df['rating'].mean()

df['rating'] = df['rating'].fillna(avg_rating)

print(df['rating'].describe())

count    1000.000000
mean        4.145729
std         0.862461
min         1.000000
25%         4.000000
50%         4.145729
75%         5.000000
max         5.000000
Name: rating, dtype: float64


**Penjelasan** : Mengunakan fillna karena agar tidak membuang baris. Jika menggunakan drop maka harus mengurangi baris kolom lain agar sama dengan baris rating agar datanya bersih. Dan mengisi dengan mean agar lebih seimbang dan merata

**C. Transformasi Data**

1. Menambahkan kolom pendapatan

In [50]:
df['total_pendapatan'] = df['unit_terjual'] * df['harga_satuan']

display(df.head())

,order_id,tanggal,kategori,kota,unit_terjual,harga_satuan,metode_pembayaran,rating,total_pendapatan
0,ORD-3000,2026-09-02,Rumah Tangga,Yogyakarta,3,90000,COD,4.0,270000
1,ORD-3001,2026-09-04,Makanan & Minuman,Solo,3,200000,E-Wallet,5.0,600000
2,ORD-3002,2026-09-26,Kesehatan & Kecantikan,Semarang,8,60000,E-Wallet,3.0,480000
3,ORD-3003,2026-09-09,Makanan & Minuman,Semarang,6,350000,Transfer Bank,4.0,2100000
4,ORD-3004,2026-09-10,Rumah Tangga,Yogyakarta,10,60000,E-Wallet,4.0,600000


2. Menambahkan kolom tier transaksi

In [51]:
import numpy as np

df['tier_transaksi'] = np.where(df['total_pendapatan'] > 500000, "Besar", "Kecil")

display(df.head())

PandasNotImplementedError: The method `pd.Series.__iter__()` is not implemented. If you want to collect your data as an NumPy array, use 'to_numpy()' instead.

Sebagai bentuk eksplorasi, dengan pyspark pandas tidak bisa menangani kasus untuk pengkondisian. Karena dalam data terdistribusi, menolak looping. Sehingga proses numpy where tidak bisa. Oleh karena itu diubah ke spark lagi dahulu

In [54]:
from pyspark.sql.functions import sum as spark_sum, count, avg, col, when

df_spark = df.to_spark()

df_spark = df_spark.withColumn(
     "tier_transaksi",
     when(col("total_pendapatan") > 500000, "Besar").otherwise("Belum Tercapai")
)

df_spark.select("tier_transaksi").show(5)

+--------------+
|tier_transaksi|
+--------------+
|Belum Tercapai|
|         Besar|
|Belum Tercapai|
|         Besar|
|         Besar|
+--------------+
only showing top 5 rows



**D. Analisis dengan Groupby**

1. kategori dengan nilai pendapatan tertinggi

In [56]:
df_spark.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("rank_pendapatan_tertinggi")
).orderBy(col("rank_pendapatan_tertinggi").desc()).show(5)

+--------------------+-------------------------+
|            kategori|rank_pendapatan_tertinggi|
+--------------------+-------------------------+
|        Rumah Tangga|                138665000|
|   Makanan & Minuman|                131890000|
|Kesehatan & Kecan...|                128595000|
|            Olahraga|                126650000|
|             Fashion|                124075000|
+--------------------+-------------------------+
only showing top 5 rows



2. kota dengan jumlah transaksi tier "besar" terbanyak

In [59]:
from pyspark.sql.functions import count_if
df_spark.groupBy("kota").agg(
    count_if(col("tier_transaksi") == "Besar").alias("tier_besar_terbanyak")
).orderBy(col("tier_besar_terbanyak").desc()).show(5)

+----------+--------------------+
|      kota|tier_besar_terbanyak|
+----------+--------------------+
|      Solo|                  92|
|  Magelang|                  78|
|   Kebumen|                  78|
|Yogyakarta|                  75|
| Purworejo|                  66|
+----------+--------------------+
only showing top 5 rows



3. rata rata rating masing masing pembayaran

In [63]:
df_spark.groupBy("metode_pembayaran").agg(
    avg(col("rating")).alias("avg_rating")
).show(5)

+-----------------+------------------+
|metode_pembayaran|        avg_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|     Kartu Kredit|4.1179474608816475|
|         E-Wallet| 4.137728643216084|
+-----------------+------------------+



**E. Menyimpan ke HDFS**

In [66]:
df_spark.write.option("header", "true").csv("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/transaksi_ecommerce_sep26")
print("Data berhasil disimpan di lokal")

Data berhasil disimpan di lokal


In [67]:
!hdfs dfs -put -f transaksi_ecommerce_sep26 /user/asfadani/tugas4/

!hdfs dfs -ls /user/asfadani/tugas4/

Found 2 items
drwxr-xr-x   - asfadani supergroup          0 2026-09-10 14:30 /user/asfadani/tugas4/transaksi_ecommerce_sep26
-rw-r--r--   1 asfadani supergroup      83497 2026-09-10 11:51 /user/asfadani/tugas4/transaksi_september_2026.csv


**Penjelasan** : hasil disimpan sebagai partisi karena spark merupakan penyimpanan terdustribusi

In [72]:
spark.stop()
print("sparksession ditutup")

sparksession ditutup


In [70]:
print("Versi Spark:", spark.version)

Versi Spark: 3.5.9
